<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_07_5_shapes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 7: PyTorch Building Blocks**  

- Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
- For more information visit the [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Module 7 Material


* Part 7.1: Model Structure [[Video]]() [[Notebook]](t81_558_class_07_1_model_structure.ipynb)
* Part 7.2: Learnable Layers [[Video]]() [[Notebook]](t81_558_class_07_2_learnable_layers.ipynb)
* Part 7.3: Nonlinearities (Activations) [[Video]]() [[Notebook]](t81_558_class_07_3_transfer.ipynb)
* Part 7.4: Normalization & Regularization [[Video]]() [[Notebook]](t81_558_class_07_4_normalization.ipynb)
* **Part 7.5: Shape** [[Video]]() [[Notebook]](t81_558_class_07_5_shapes.ipynb)



# Google CoLab Instructions

The following code checks that Google CoLab is and sets up the correct hardware settings for PyTorch.


In [ ]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 2.5)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Part 7.5: Shape

Shape and resolution transforms adjust the structure of data as it moves through the network. Pooling layers reduce spatial or temporal dimensions, which can improve efficiency and capture broader patterns. Reshaping operations such as flattening prepare data for transitions between different layer types. These transformations ensure compatibility between components.

Unlike the learnable layers introduced in Part 7.2, shape transforms generally have no trainable parameters. They do not learn from data. Instead, they perform fixed operations that change how a tensor is organized, what dimensions it has, or how much information it carries. Despite their simplicity, these operations are essential. A convolutional feature map cannot feed directly into a linear classifier without reshaping, and a network that preserves full spatial resolution at every layer would be prohibitively expensive to train. Shape transforms are the connective tissue that holds an architecture together.

## Why Shape Matters in PyTorch

Every tensor in PyTorch carries a shape that describes how its values are arranged across dimensions. A grayscale image might be stored as a tensor with shape (1, 28, 28), representing one channel and a 28 by 28 grid of pixels. A batch of 64 such images takes the shape (64, 1, 28, 28), where the leading dimension is the batch size. As data flows through a network, each layer expects its input to have a specific shape and produces output with a predictable new shape.

When a layer receives a tensor of the wrong shape, PyTorch raises a runtime error rather than silently producing incorrect results. This strictness is helpful during development because it surfaces structural mistakes immediately. However, it also means that the architect of a network must reason carefully about how shapes evolve from input to output. Pooling and reshaping operations are the primary tools for managing this evolution.

## Pooling Layers

Pooling layers reduce the size of a tensor along its spatial or temporal dimensions while leaving the channel or feature dimension untouched. The most common variants are max pooling and average pooling. Max pooling slides a small window across the input and keeps only the largest value within each window. Average pooling slides the same window but reports the mean of the values it covers. Both operations downsample the data, producing a smaller tensor that summarizes local regions of the original.

Pooling serves two purposes. First, it reduces computational cost. A 2 by 2 pooling operation with a stride of 2 cuts the spatial extent of a feature map in half along each axis, which reduces the number of values by a factor of four. Subsequent layers operate on a smaller tensor and therefore require less memory and fewer multiplications. Second, pooling encourages the network to focus on broader patterns. By summarizing a neighborhood with a single value, the layer becomes less sensitive to small translations or distortions in the input. A feature that appears slightly to the left or right of where the network expected it will still register in the pooled output.

PyTorch provides pooling modules for one, two, and three dimensional data. The class `nn.MaxPool2d` is the standard choice for image inputs, while `nn.MaxPool1d` is appropriate for sequence data and `nn.MaxPool3d` for volumetric data such as medical scans. Adaptive variants, including `nn.AdaptiveAvgPool2d`, allow the user to specify the desired output size and let PyTorch compute the necessary window and stride internally. Adaptive pooling is particularly useful when the input size varies across examples, since it guarantees a fixed output shape regardless of the input.

## Flattening and Reshaping

Flattening collapses several dimensions of a tensor into a single dimension. This operation is most often used at the boundary between convolutional layers and fully connected layers. A convolutional stack produces a feature map with shape (batch, channels, height, width), but a linear layer expects an input of shape (batch, features). Flattening every dimension after the batch dimension converts the four dimensional tensor into a two dimensional tensor that the linear layer can consume.

PyTorch offers several ways to reshape a tensor. The `nn.Flatten` module is convenient inside a `Sequential` block because it behaves like any other layer. The tensor methods `view` and `reshape` provide finer control when shape manipulation happens in a custom `forward` method. The `view` method requires the underlying memory to be contiguous, while `reshape` will copy the data if necessary. For most purposes the difference is invisible, but it becomes relevant when working with tensors that have been transposed or otherwise made non contiguous.

Reshaping is a logical operation rather than a numerical one. The values in the tensor do not change, only the way those values are indexed. This makes reshaping very fast, but it also means the user must ensure that the new shape contains the same total number of elements as the old shape. PyTorch will raise an error if the totals do not match.

## A Practical Example

The following example builds a small convolutional network that classifies 28 by 28 grayscale images. The network applies two convolutional blocks, each followed by max pooling, then flattens the resulting feature map and passes it through a linear layer. At each stage the code prints the shape of the tensor so that the effect of every transform is visible.

In [ ]:
import torch
import torch.nn as nn


class ShapeDemoNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(in_features=32 * 7 * 7, out_features=num_classes)

    def forward(self, x):
        print(f"Input shape:            {tuple(x.shape)}")

        x = self.relu(self.conv1(x))
        print(f"After first conv:       {tuple(x.shape)}")
        x = self.pool(x)
        print(f"After first pool:       {tuple(x.shape)}")

        x = self.relu(self.conv2(x))
        print(f"After second conv:      {tuple(x.shape)}")
        x = self.pool(x)
        print(f"After second pool:      {tuple(x.shape)}")

        x = self.flatten(x)
        print(f"After flatten:          {tuple(x.shape)}")

        x = self.fc(x)
        print(f"After linear:           {tuple(x.shape)}")
        return x


# Create a batch of 4 fake grayscale images, each 28 by 28 pixels.
batch = torch.randn(4, 1, 28, 28)

model = ShapeDemoNet(num_classes=10)
logits = model(batch)
print(f"\nFinal output represents class scores for {logits.shape[0]} examples.")

Tracing the printed shapes reveals the role of each transform. The convolutional layers preserve the 28 by 28 spatial dimensions because of the padding setting, but they grow the channel dimension from 1 to 16 and then to 32. The pooling layers leave the channel count alone but cut the height and width in half each time, taking the resolution from 28 down to 14 and finally to 7. Flattening collapses the channel, height, and width dimensions into a single feature vector of length 32 times 7 times 7, which equals 1568. The linear layer then maps that vector to 10 class scores. Pooling and flattening together carry the data from a four dimensional image representation to a one dimensional class prediction without any of them learning a single parameter.